In [1]:
import pandas as pd
import re
rep="/Volumes/BroadExt/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/"

In [2]:
fParadigms="vlexique_paradigms.csv"
fFeatures="vlexique_features.csv"
fLexemes="vlexique_lexemes.csv"
fSounds="vlexique_sounds.csv"
fTags="vlexique_tags.csv"
fCells="vlexique_cells.csv"

dfParadigms=pd.read_csv(rep+fParadigms)
dfFeatures=pd.read_csv(rep+fFeatures)
dfLexemes=pd.read_csv(rep+fLexemes)
dfSounds=pd.read_csv(rep+fSounds)
dfTags=pd.read_csv(rep+fTags)
dfCells=pd.read_csv(rep+fCells)

In [3]:
dGrace=dfCells[["cell_id","GRACE"]].set_index("cell_id").to_dict()["GRACE"]
dSwim={}
for k,v in dGrace.items():
    # pour les formes finies, on a un matching m
    m=re.match("Vm(.)(.)([123][sp])-",v)
    if m:
        # le passé simple est codé s en GRACE mais a pour SWIM
        # l'impératif est codé m en GRACE mais I pour SWIM
        dSwim[k]=m.group(2).replace("s","a")+m.group(1).replace("m","I")+m.group(3).upper()
    else:
        m=re.match("Vmp(.)-(.)(.)",v)
        if m:
            if m.group(1)=="p":
                dSwim[k]="pP"
            else:
                dSwim[k]="pp"+m.group(3).upper()+m.group(2).upper()
        else:
            dSwim[k]="inf"
dSwim

{'cond.prs.1.pl': 'pc1P',
 'cond.prs.1.sg': 'pc1S',
 'cond.prs.2.pl': 'pc2P',
 'cond.prs.2.sg': 'pc2S',
 'cond.prs.3.pl': 'pc3P',
 'cond.prs.3.sg': 'pc3S',
 'imp.prs.1.pl': 'pI1P',
 'imp.prs.2.pl': 'pI2P',
 'imp.prs.2.sg': 'pI2S',
 'ind.fut.1.pl': 'fi1P',
 'ind.fut.1.sg': 'fi1S',
 'ind.fut.2.pl': 'fi2P',
 'ind.fut.2.sg': 'fi2S',
 'ind.fut.3.pl': 'fi3P',
 'ind.fut.3.sg': 'fi3S',
 'ind.ipfv.1.pl': 'ii1P',
 'ind.ipfv.1.sg': 'ii1S',
 'ind.ipfv.2.pl': 'ii2P',
 'ind.ipfv.2.sg': 'ii2S',
 'ind.ipfv.3.pl': 'ii3P',
 'ind.ipfv.3.sg': 'ii3S',
 'ind.prs.1.pl': 'pi1P',
 'ind.prs.1.sg': 'pi1S',
 'ind.prs.2.pl': 'pi2P',
 'ind.prs.2.sg': 'pi2S',
 'ind.prs.3.pl': 'pi3P',
 'ind.prs.3.sg': 'pi3S',
 'ind.pst.1.pl': 'ai1P',
 'ind.pst.1.sg': 'ai1S',
 'ind.pst.2.pl': 'ai2P',
 'ind.pst.2.sg': 'ai2S',
 'ind.pst.3.pl': 'ai3P',
 'ind.pst.3.sg': 'ai3S',
 'inf': 'inf',
 'ptcp.prs': 'pP',
 'ptcp.pst.f.pl': 'ppFP',
 'ptcp.pst.f.sg': 'ppFS',
 'ptcp.pst.m.pl': 'ppMP',
 'ptcp.pst.m.sg': 'ppMS',
 'sbjv.prs.1.pl': 'ps1P',

In [4]:
sounds=dfSounds.sound_id.unique().tolist()
print(", ".join(sounds))

p, b, t, d, k, g, f, v, s, z, ʃ, ʒ, m, n, ɲ, ŋ, ʁ, l, j, w, ɥ, i, y, u, E, e, ɛ, Ø, ə, ø, œ, O, o, ɔ, a, ɛ̃, œ̃, ɑ̃, ɔ̃


In [5]:
dSound={"ʃ":"S","ʒ":"Z","ɲ":"J","ŋ":"N","ʁ":"r","ɥ":"H",
        "ɛ":"E","Ø":"2","ə":"6","ø":"2","œ":"9","ɔ":"O",
        "ɛ̃":"ê","œ̃":"û","ɑ̃":"â","ɔ̃":"ô"}

def mapSounds(chaine):
    result=chaine
    result=result.replace("ɛ̃","ê")
    result=result.replace("œ̃","û")
    result=result.replace("ɑ̃","â")
    result=result.replace("ɔ̃","ô")
    for sound in dSound:
        result=result.replace(sound,dSound[sound])
    return result.replace(" ","")

In [6]:
mapSounds("a b ɛ s ə ʁ j ɔ̃")

'abEs6rjô'

In [7]:
specifics=dfParadigms.loc[(dfParadigms.overabundance_tag=="specific")]["lexeme"].unique().tolist()

# corrections 2.0.3 pour asseoir et rasseoir
dfParadigms.loc[(dfParadigms.lexeme=="asseoir")&(dfParadigms.orth_form.str.contains("oi"))&(~dfParadigms.phon_form.str.contains("w a")),"phon_form"]="a s w a"
dfParadigms.loc[(dfParadigms.lexeme=="rasseoir")&(dfParadigms.orth_form.str.contains("oi"))&(~dfParadigms.phon_form.str.contains("w a")),"phon_form"]="r a s w a"

# corrections 2.0.3 pour arguer
dfParadigms.loc[(dfParadigms.lexeme=="arguer")&(dfParadigms.phon_form.str.contains("u")),"phon_form"]=dfParadigms.loc[(dfParadigms.lexeme=="arguer")&(dfParadigms.phon_form.str.contains("u")),"phon_form"].str.replace("u","y")

# corrections 2.0.3 pour catir
dfParadigms.loc[(dfParadigms.lexeme=="catir")&(dfParadigms.cell=="ind.prs.2.pl"),"orth_form"]="catissez"

In [8]:
dfParadigms.loc[:,"sPhon"]=dfParadigms.phon_form.apply(mapSounds)

In [9]:
print(dfParadigms.loc[dfParadigms.lexeme.isin(specifics)].groupby(["lexeme","cell"])[["orth_form","sPhon"]].agg(set).to_string())

                                                        orth_form                         sPhon
lexeme     cell                                                                                
arguer     cond.prs.1.pl     {arguërions, arguerions, argüerions}           {arg6rjô, argy6rjô}
           cond.prs.1.sg        {arguerais, arguërais, argüerais}             {argy6rE, arg6rE}
           cond.prs.2.pl        {arguëriez, argueriez, argüeriez}           {arg6rje, argy6rje}
           cond.prs.2.sg        {arguerais, arguërais, argüerais}             {argy6rE, arg6rE}
           cond.prs.3.pl  {argueraient, arguëraient, argüeraient}             {argy6rE, arg6rE}
           cond.prs.3.sg        {argüerait, arguërait, arguerait}             {argy6rE, arg6rE}
           imp.prs.1.pl                        {argüons, arguons}                 {argyô, argô}
           imp.prs.2.pl                          {argüez, arguez}                 {argye, arge}
           imp.prs.2.sg                 

In [10]:
df=dfParadigms[["orth_form","phon_form","lexeme","frequency","cell"]]
df.loc[:,"gCell"]=df["cell"].replace(dGrace)
df.loc[:,"sCell"]=df["cell"].replace(dSwim)
df.loc[:,"sPhon"]=df.phon_form.apply(mapSounds)
df

/var/folders/dy/h2nycthd7qjbd6qkb5h8t0nm0000gn/T/ipykernel_34672/298172186.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,"gCell"]=df["cell"].replace(dGrace)
/var/folders/dy/h2nycthd7qjbd6qkb5h8t0nm0000gn/T/ipykernel_34672/298172186.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:,"sCell"]=df["cell"].replace(dSwim)
/var/folders/dy/h2nycthd7qjbd6qkb5h8t0nm0000gn/T/ipykernel_34672/298172186.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFra

,orth_form,phon_form,lexeme,frequency,cell,gCell,sCell,sPhon
0,abaisserions,a b ɛ s ə ʁ j ɔ̃,abaisser,0,cond.prs.1.pl,Vmcp1p-,pc1P,abEs6rjô
1,abaisserais,a b ɛ s ə ʁ E,abaisser,38,cond.prs.1.sg,Vmcp1s-,pc1S,abEs6rE
2,abaisseriez,a b ɛ s ə ʁ j e,abaisser,9,cond.prs.2.pl,Vmcp2p-,pc2P,abEs6rje
3,abaisserais,a b ɛ s ə ʁ E,abaisser,14,cond.prs.2.sg,Vmcp2s-,pc2S,abEs6rE
4,abaisseraient,a b ɛ s ə ʁ E,abaisser,4,cond.prs.3.pl,Vmcp3p-,pc3P,abEs6rE
...,...,...,...,...,...,...,...,...
274891,sois,s w a,être,15983,sbjv.prs.1.sg,Vmsp1s-,ps1S,swa
274892,soyez,s w a j e,être,13798,sbjv.prs.2.pl,Vmsp2p-,ps2P,swaje
274893,sois,s w a,être,26590,sbjv.prs.2.sg,Vmsp2s-,ps2S,swa
274894,soient,s w a,être,21743,sbjv.prs.3.pl,Vmsp3p-,ps3P,swa


In [11]:
dfSelect=df[["orth_form","sPhon","lexeme","frequency","sCell"]]
dfSelect.columns=["ortho","phono","lexeme","tir1","case"]
dfSelect.loc[(dfSelect.lexeme=="asseoir") & (dfSelect.case.str.startswith("ps")),:]

,ortho,phono,lexeme,tir1,case
16258,asseyions,asEjô,asseoir,6,ps1P
16259,assoyions,aswajô,asseoir,1,ps1P
16260,asseye,asEj,asseoir,59,ps1S
16261,assoie,aswa,asseoir,130,ps1S
16262,asseyiez,asEje,asseoir,37,ps2P
16263,assoyiez,aswaje,asseoir,3,ps2P
16264,asseyes,asEj,asseoir,33,ps2S
16265,assoies,aswa,asseoir,70,ps2S
16266,asseyent,asEj,asseoir,11,ps3P
16267,assoient,aswa,asseoir,36,ps3P


In [12]:
dfOut=dfSelect.groupby(["phono","lexeme","case"])[["ortho","tir1"]].agg({"ortho":list,"tir1":"sum"}).reset_index()
dfOut["ortho"]=dfOut["ortho"].str.join(",")
dfOut.loc[dfOut.lexeme=="abréger",:]

,phono,lexeme,case,ortho,tir1
30752,abrEZ,abréger,pI2S,abrège,95
30753,abrEZ,abréger,pi1S,abrège,45
30754,abrEZ,abréger,pi2S,abrèges,6
30755,abrEZ,abréger,pi3P,abrègent,9
30756,abrEZ,abréger,pi3S,abrège,35
30757,abrEZ,abréger,ps1S,abrège,10
30758,abrEZ,abréger,ps2S,abrèges,0
30759,abrEZ,abréger,ps3P,abrègent,1
30760,abrEZ,abréger,ps3S,abrège,17
30761,abrEZ6rE,abréger,fi1S,"abrègerai,abrégerai",13


In [13]:
# dfOut.to_csv(rep+"vlexique2-tir.csv",sep="\t",encoding="utf8")
dfOut.to_pickle(rep+"vlexique2.pkl")

In [15]:
print(dfOut.loc[dfOut.case=="ai3P",:].to_string())

                    phono               lexeme  case                            ortho  tir1
4                   #DEF#             absoudre  ai3P                       absolurent     0
16                  #DEF#            abstraire  ai3P                     abstrayèrent     0
28                  #DEF#             accroire  ai3P                            #DEF#     0
109                 #DEF#                ardre  ai3P                            #DEF#     0
153                 #DEF#               braire  ai3P                        brayèrent     0
188                 #DEF#              chaloir  ai3P                            #DEF#     0
255                 #DEF#                clore  ai3P                            #DEF#     0
278                 #DEF#            comparoir  ai3P                            #DEF#     0
327                 #DEF#         discontinuer  ai3P                  discontinuèrent     0
377                 #DEF#            dissoudre  ai3P                      dissol